In [1]:
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv('QVI_data_task2.csv')

In [6]:
df.head(5)

,LYLTY_CARD_NBR,DATE,STORE_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
0,1000,2018-10-17,1,1,5,Natural Chip Compny SeaSalt175g,2,6.0,175,NATURAL,YOUNG SINGLES/COUPLES,Premium
1,1002,2018-09-16,1,2,58,Red Rock Deli Chikn&Garlic Aioli 150g,1,2.7,150,RRD,YOUNG SINGLES/COUPLES,Mainstream
2,1003,2019-03-07,1,3,52,Grain Waves Sour Cream&Chives 210G,1,3.6,210,GRNWVES,YOUNG FAMILIES,Budget
3,1003,2019-03-08,1,4,106,Natural ChipCo Hony Soy Chckn175g,1,3.0,175,NATURAL,YOUNG FAMILIES,Budget
4,1004,2018-11-02,1,5,96,WW Original Stacked Chips 160g,1,1.9,160,WOOLWORTHS,OLDER SINGLES/COUPLES,Mainstream


In [7]:
df.dtypes

LYLTY_CARD_NBR        int64
DATE                 object
STORE_NBR             int64
TXN_ID                int64
PROD_NBR              int64
PROD_NAME            object
PROD_QTY              int64
TOT_SALES           float64
PACK_SIZE             int64
BRAND                object
LIFESTAGE            object
PREMIUM_CUSTOMER     object
dtype: object

In [8]:
# Create YearMonth identifier
df['YEARMONTH'] = df['DATE'].apply(lambda x: int(x.replace('-', '')[:6]))

# Aggregate monthly metrics per store
measure_over_time = df.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    totSales=('TOT_SALES', 'sum'),
    nCustomers=('LYLTY_CARD_NBR', 'nunique'),
    nTxns=('TXN_ID', 'nunique')
).reset_index()

# Calculate Transactions per Customer
measure_over_time['nTxnPerCust'] = measure_over_time['nTxns'] / measure_over_time['nCustomers']

# Filter for stores with a full 12 months of data
store_counts = measure_over_time.groupby('STORE_NBR').size()
stores_with_full_obs = store_counts[store_counts == 12].index
measure_over_time = measure_over_time[measure_over_time['STORE_NBR'].isin(stores_with_full_obs)]

In [9]:
def calculate_correlation(store1, store2, df, metric):
    """Calculates Pearson correlation between two stores for a specific metric."""
    s1 = df[df['STORE_NBR'] == store1][metric]
    s2 = df[df['STORE_NBR'] == store2][metric]
    return s1.corr(s2)

def calculate_magnitude_distance(store1, store2, df, metric):
    """Calculates standardized magnitude distance (1 - normalized distance)."""
    dist = abs(df[df['STORE_NBR'] == store1][metric].mean() - df[df['STORE_NBR'] == store2][metric].mean())
    # Normalize distance to a 0-1 scale where 1 is most similar
    return 1 - (dist / df[metric].max())

# Example: Finding a match for Store 77
pre_trial = measure_over_time[measure_over_time['YEARMONTH'] < 201902]
other_stores = pre_trial[pre_trial['STORE_NBR'] != 77]['STORE_NBR'].unique()

scores = []
for store in other_stores:
    corr_score = calculate_correlation(77, store, pre_trial, 'totSales')
    mag_score = calculate_magnitude_distance(77, store, pre_trial, 'totSales')
    scores.append({'Store': store, 'Score': (corr_score + mag_score) / 2})

# View top 5 potential matches
pd.DataFrame(scores).sort_values('Score', ascending=False).head(5)

,Store,Score
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN


In [13]:
def calculate_correlation(store1, store2, df, metric):
    # Filter for the two stores
    s1 = df[df['STORE_NBR'] == store1].set_index('YEARMONTH')[metric]
    s2 = df[df['STORE_NBR'] == store2].set_index('YEARMONTH')[metric]
    # Use pandas series correlation which aligns by the YEARMONTH index
    return s1.corr(s2)

def calculate_magnitude_distance(store1, store2, df, metric):
    dist = abs(df[df['STORE_NBR'] == store1][metric].mean() - df[df['STORE_NBR'] == store2][metric].mean())
    # Magnitude should be relative to the range of the metric
    max_dist = df[metric].max() - df[metric].min()
    return 1 - (dist / max_dist)

# Re-run matching for Store 77
scores = []
for store in valid_control_stores:
    if store == 77: continue
    
    corr_score = calculate_correlation(77, store, pre_trial, 'totSales')
    mag_score = calculate_magnitude_distance(77, store, pre_trial, 'totSales')
    
    if not np.isnan(corr_score):
        # We give 50% weight to correlation and 50% to magnitude
        scores.append({'Store': store, 'Score': (corr_score + mag_score) / 2})

if scores:
    results_df = pd.DataFrame(scores).sort_values('Score', ascending=False)
    print("Top matches for Store 77:")
    print(results_df.head(5))
else:
    print("No valid matches found. Check if 'pre_trial' contains data for Store 77.")

Top matches for Store 77:
     Store     Score
220    233  0.950197
38      41  0.877933
15      17  0.874375
46      50  0.873458
107    115  0.818127
